# 🧪 Sesión 39 · Monitoreo, Métricas de Salud (/health) y Profiling Web

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo sabe un balanceador si tu API está sana?

---

## 🖥️ Así se verá el resultado final

```http
GET /health
HTTP/1.1 200 OK
{"status":"ok","db":"ok","version":"1.0.0"}

GET /health?simular_fallo=true
HTTP/1.1 503 Service Unavailable
{"status":"error","db":"error","version":"1.0.0"}

GET /api/metricas
HTTP/1.1 200 OK
{"uptime":12.4,"version":"1.0.0"}
```

_Un healthcheck debe ser barato y comprobar las dependencias críticas, no solo que el proceso exista._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Exponer un endpoint de salud con dependencias.
- Responder 503 cuando la base no está disponible.
- Verificar el contrato de monitoreo.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El servicio sano responde 200 → HTTP 200
- [ ] El fallo simulado responde 503 → HTTP 503
- [ ] Las métricas están disponibles → HTTP 200

## 🧩 El reto

Implementa un endpoint de salud con verificación de base de datos y un endpoint de métricas básicas.

**Requisitos del contrato**

- GET /health responde 200 con status ok, db ok y version.
- GET /health?simular_fallo=true responde 503 con status error.
- GET /api/metricas responde 200 con uptime y version.
- El healthcheck consulta la base de datos, no solo el proceso.
- La respuesta no expone secretos ni rutas internas.

> **Pista conceptual:** El healthcheck no debe ser costoso. Un `SELECT 1` a la base es suficiente para readiness; no recorras tablas completas.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion39.md` con las capturas de 200 y 503.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
import time

from fastapi import FastAPI
from fastapi.responses import JSONResponse

app = FastAPI(title="Tu healthcheck se verá así")

VERSION = "1.0.0"
INICIO = time.time()


def db_disponible(simular_fallo: bool) -> bool:
    """Simula la comprobación de la base de datos."""
    return not simular_fallo


@app.get("/health")
def health(simular_fallo: bool = False):
    if db_disponible(simular_fallo):
        return {"status": "ok", "db": "ok"}
    return JSONResponse(status_code=503, content={"status": "error", "db": "error"})


@app.get("/api/metricas")
def metricas():
    return {
        "version": VERSION,
        "uptime_segundos": round(time.time() - INICIO, 2),
    }


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'El servicio sano responde 200',
  'metodo': 'GET',
  'ruta': '/health',
  'estado': 200,
  'json_contiene': {'status': 'ok', 'db': 'ok'}},
 {'descripcion': 'El fallo simulado responde 503',
  'metodo': 'GET',
  'ruta': '/health?simular_fallo=true',
  'estado': 503,
  'json_contiene': {'status': 'error', 'db': 'error'}},
 {'descripcion': 'Las métricas están disponibles',
  'metodo': 'GET',
  'ruta': '/api/metricas',
  'estado': 200,
  'json_contiene': {'version': '1.0.0'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. El servicio sano responde 200
✅ 02. El fallo simulado responde 503
✅ 03. Las métricas están disponibles
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion39.md` a `auditorias/auditoria_sesion39.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
